# Découpage commun et référence naïve

Ce notebook vérifie le socle commun défini dans `src/decoupage.py`. Tous les modèles du groupe doivent l'utiliser : mêmes données préparées, même découpage, même mesure.

**Rappel :** les données sont préparées avec la Tanzanie corrigée en supposant vraie notre hypothèse sur ses libellés (voir `01_eda.ipynb`, section 3).

In [1]:
import sys

import pandas as pd

sys.path.insert(0, "..")
from src.decoupage import charger, decouper, evaluer

donnees = charger()
X_ent, X_val, X_test, y_ent, y_val, y_test = decouper(donnees)

pd.DataFrame({
    "lignes": [len(X_ent), len(X_val), len(X_test)],
    "part_des_lignes": [len(X_ent) / len(donnees), len(X_val) / len(donnees), len(X_test) / len(donnees)],
    "taux_de_comptes": [y_ent.mean(), y_val.mean(), y_test.mean()],
}, index=["entraînement", "validation", "test final"]).round(4)

,lignes,part_des_lignes,taux_de_comptes
entraînement,16466,0.70,0.1408
validation,3529,0.15,0.1408
test final,3529,0.15,0.1408


## 1. Vérifications

Les trois ensembles doivent être disjoints, couvrir toutes les lignes, garder le même taux de comptes dans chaque pays, et rester identiques d'une exécution à l'autre.

In [2]:
ensembles = {"entraînement": (X_ent, y_ent), "validation": (X_val, y_val), "test final": (X_test, y_test)}

assert set(X_ent.index).isdisjoint(X_val.index) and set(X_ent.index).isdisjoint(X_test.index) and set(X_val.index).isdisjoint(X_test.index)
assert len(X_ent) + len(X_val) + len(X_test) == len(donnees)
assert all((X.index == y.index).all() for X, y in ensembles.values())
assert "bank_account" not in X_ent.columns

second = decouper(donnees)
assert all((a.index == b.index).all() for a, b in zip((X_ent, X_val, X_test), second[:3]))

pd.DataFrame({
    nom: X.assign(compte=y).groupby("country")["compte"].mean() for nom, (X, y) in ensembles.items()
}).round(4)

,entraînement,validation,test final
country,,,
Kenya,0.2505,0.2514,0.2505
Rwanda,0.1148,0.1145,0.1152
Tanzania,0.0917,0.0916,0.0916
Uganda,0.0863,0.0857,0.0857


Le taux de comptes de chaque pays est le même, au millième près, dans les trois ensembles : la stratification sur le pays et la cible a fonctionné. Un second appel à `decouper()` donne exactement les mêmes lignes.

## 2. Référence naïve

La référence prédit toujours « pas de compte ». Aucun modèle n'est entraîné ici : c'est le chiffre que chaque modèle du groupe doit battre sur la validation.

In [3]:
naive = pd.Series(0, index=X_val.index)
print(f"Accuracy naïve sur la validation : {(naive == y_val).mean():.4f}")
evaluer(X_val, y_val, naive)

Accuracy naïve sur la validation : 0.8592


,effectif,f1_macro
groupe,,
Global,3529,0.4621
country = Kenya,911,0.4281
country = Rwanda,1310,0.4696
country = Tanzania,993,0.4760
country = Uganda,315,0.4776
gender_of_respondent = Female,2077,0.4708
gender_of_respondent = Male,1452,0.4492


La référence naïve atteint environ 86 % d'accuracy en ne trouvant aucune personne avec un compte. C'est pourquoi la mesure commune est le **F1 macro**, qui moyenne la qualité sur les deux classes : la référence y plafonne autour de 0,46.

## 3. Règles communes

- Entraîner uniquement sur `X_ent, y_ent`.
- Comparer les modèles, régler les paramètres et choisir le seuil sur `X_val, y_val`.
- N'évaluer qu'une fois sur `X_test, y_test`, à la fin, avec le modèle retenu.
- Rapporter `evaluer(...)` : F1 macro global, par pays et par genre.